In [5]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver import Keys
import requests

# 폴더 관리를 위한 모듈
import os

# 이미지 링크 주소 파싱을 위한 모듈
from urllib.parse import urlparse, unquote

In [6]:
driver = webdriver.Chrome()

In [7]:
url = 'https://www.google.com/imghp?hl=ko&ogbl'
driver.get(url)

In [4]:
# 검색어 입력 및 검색
img_name = '윈터'
search_img = driver.find_element(By.CSS_SELECTOR, 'textarea.gLFyf')
search_img.send_keys(img_name)
driver.implicitly_wait(1)           # 암시적 대기 1초
search_img.send_keys(Keys.ENTER)

In [93]:
# 처음 이미지 선택(선택해서 클릭한 이미지가 원본 이미지)
# img 값이 변하므로 확인 필수!! -> 다른 방법 강구
first_img = driver.find_element(By.CSS_SELECTOR, 'div#search h3.ob5Hkd img.YQ4gaf')
first_img.click()
driver.implicitly_wait(2)    

In [94]:
# 원본 이미지 링크 확인
target_img = driver.find_element(By.CSS_SELECTOR, "img.sFlh5c.FyHeAf.iPVvYb")
target_img_link = target_img.get_attribute('src')
print(target_img_link)

https://menu.moneys.co.kr/moneyweek/thumb/2025/01/10/06/2025011016581868696_1.jpg


In [95]:
# 목표 사진 개수 설정 및 저장 폴더 설정
target_count = 20
save_dir = f'./{img_name}/'

if not os.path.exists(save_dir):
    os.makedirs(save_dir, exist_ok=True)

print(f'{save_dir} 폴더가 생성되었거나 이미 존재합니다.')

./윈터/ 폴더가 생성되었거나 이미 존재합니다.


In [96]:
# 버튼 테스트
next_div = driver.find_elements(By.CSS_SELECTOR, 'div.HJRshd')

target_div = next_div[-2]

buttons = target_div.find_elements(By.TAG_NAME, 'button')

# 버튼 확인용
for btn in buttons:
    print(btn.get_attribute('aria-label'))

이전 이미지
다음 이미지
이 검색 결과에 관한 작업 더보기
닫기


In [97]:
# 이미지 저장 링크 테스트
target_img = driver.find_elements(By.CSS_SELECTOR, "img.sFlh5c.FyHeAf.iPVvYb")
for img in target_img:
    print("이미지 링크입니다.", img.get_attribute("src"))

이미지 링크입니다. https://menu.moneys.co.kr/moneyweek/thumb/2025/01/10/06/2025011016581868696_1.jpg


In [98]:
# 이미지 저장 테스트 _ 이미지 데이터를 GET 요청으로 가져오기
target_img_link = target_img[-1].get_attribute('src')
response = requests.get(target_img_link)

# 요청이 성공적인 겨우, 이미지를 로컬 파일로 저장
if response.status_code == 200:
    path = urlparse(target_img_link).path
    decoded_path = unquote(path)
    file_name = decoded_path.split("/")[-1]
    file_ext = file_name.split('.')[-1]
    save_name = f'{save_dir}{img_name}_test.{file_ext}'

    with open(save_name,'wb') as file:
        file.write(response.content)
    print(f"이미지 다운로드 성공:{save_name}")
else:
    print("이미지 다운로드 실패")

이미지 다운로드 성공:./윈터/윈터_test.jpg


In [99]:
# 최종 :  다음 이미지 클릭하여, 원본 이미지 저장
for count in range(target_count):
    # 다음 이미지 버튼 요소 가져오기
    next_div = driver.find_elements(By.CSS_SELECTOR, "div.HJRshd")
    target_div = next_div[-2] 

    # 다음 이미지 버튼 클릭
    buttons = target_div.find_elements(By.TAG_NAME, "button")
    next_btn = buttons[1]
    next_btn.click()

    # target 이미지 선택
    target_img = driver.find_elements(By.CSS_SELECTOR, "img.sFlh5c.FyHeAf.iPVvYb")[-1]
    target_img_link = target_img.get_attribute("src")
    print(target_img_link)

    # 이미지 데이터를 get 요청으로 가져오기
    # 일부 웹 서버는 User-Agent가 없는 요청을 차단할 수 있음
    headers = {
        'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/58.0.3029.110 Safari/537.3'
    }
    response = requests.get(target_img_link, headers=headers, verify=False)
    
    # 요청이 성공적인 경우, 이미지를 로컬 파일로 저장
    if response.status_code == 200:
        # 이미지 확장자 확인 및 쿼리 파라미터 제거를 위한 작업
        path = urlparse(target_img_link).path
        decoded_path = unquote(path)
        file_name = decoded_path.split("/")[-1]
        file_ext = file_name.split(".")[-1]
        save_name = f"{save_dir}{img_name}_{count}.{file_ext}"
        # 바이너리 쓰기 모드로 파일 열기
        with open(save_name, 'wb') as file:
            file.write(response.content)
        print(f"이미지가 성공적으로 다운로드되었습니다: {save_name}")
    else:
        print(f"{response.status_code}_이미지 다운로드 실패")
    driver.implicitly_wait(1)   # 1초 대기 : 너무 빠른 요청을 방지하고 안정적으로 작동을 위해 대기


https://menu.moneys.co.kr/moneyweek/thumb/2025/01/10/06/2025011016581868696_1.jpg
이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_0.jpg
https://img.hankyung.com/photo/202507/03.41241898.1.jpg


c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'menu.moneys.co.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'img.hankyung.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_1.jpg
https://thumbnews.nateimg.co.kr/view610///news.nateimg.co.kr/orgImg/jn/2024/02/18/10bc0d844039f8.jpg
이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_2.jpg
https://i.namu.wiki/i/JFgk6yG9xDH15Lj1gfms1DkB7tEJPRgTvuZ2klGEb8mfnmAAt2mlpJDfC2xcGuuSrHbiJX5eOJ5SeUVNE2kdig.webp


c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'thumbnews.nateimg.co.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'i.namu.wiki'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_3.webp
https://thumb.mt.co.kr/06/2025/01/2025011021253750756_1.jpg
이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_4.jpg


...

이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_15.jpg
https://cdn.autotribune.co.kr/news/photo/202412/33088_210083_5440.jpg


c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'cdn.autotribune.co.kr'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_16.jpg
https://i.namu.wiki/i/XZDHBXlbYfwEQwcLmda2A9W8R_IIVyCdaAX2PqETJc3jsnnifMhhhvPPNmc96phxpqZtgdgeVV2obvzr1e-G5Q.webp
이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_17.webp
https://photo.newsen.com/news_photo/2023/08/02/202308021551233510_1.jpg
이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_18.jpg
https://img.sportsworldi.com/content/image/2024/06/03/20240603509255.jpg


c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'photo.newsen.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
c:\Users\Admin\miniconda3\envs\crawling\Lib\site-packages\urllib3\connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'img.sportsworldi.com'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


이미지가 성공적으로 다운로드되었습니다: ./윈터/윈터_19.jpg
